# Deliverable 4 — The Cross-Dataset Investigation

**The climax.** Congress trades + corporate lobbying + federal contract awards, joined on ticker within aligned time windows. The question is no longer *"what should I buy?"* — it's *"what should I be paying attention to?"*

**Method (scoped, per the project plan):**
1. Committee membership is joined rigorously via `BioGuideID` from the public-domain [unitedstates/congress-legislators](https://github.com/unitedstates/congress-legislators) dataset — no hand-typed rosters.
2. The candidate universe is sector-scoped where federal money is densest: defense, pharma, energy, federal tech.
3. Every Congress **buy** of a candidate ticker is scored on **time alignment** (buy → the largest contract award in the following 180 days), **dollar size** (contracts, lobbying, trade), and **committee relevance** (does the buyer sit on a committee with jurisdiction over the company's sector?).
4. Award timing is keyed on USASpending's **`action_date`** — the date the contract action actually happened. Quiver's `Date` column is its publish/ingest date (its 2022 rows are a multi-year backfill), so timing keyed on `Date` would measure Quiver's upload schedule, not contract timing.
5. The ranked candidate list is written to `candidates.csv` **for manual verification against the public record** (Senate LDA database, USASpending.gov, the actual disclosure) before anything goes in the script.

**Confidence framing:** a high score here is *"look at this closely"*, never *"this is corruption."* Everything shown is legal and public.

Outputs → `output/4-cross-dataset-investigation/`: `hero_timeline_<T1>.mp4`, `second_timeline_<T2>.mp4`, `candidates.csv`, `summary_stats.json`

In [1]:
import sys
import json
import pickle
import warnings
from pathlib import Path
from datetime import timedelta
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import requests
import yaml
import matplotlib.pyplot as plt
import matplotlib.animation as mpl_animation
import matplotlib.dates as mdates

REPO_ROOT = Path.cwd().resolve().parent
load_dotenv(REPO_ROOT / '.env')
sys.path.insert(0, str(REPO_ROOT))

from lib import brand, animation, data_quiver, data_massive
from lib.brand import BG, GREEN, OFF_WHITE, RED, OLIVE, GRID, VCR
from audio_engine import Cue, render_track, ticks_every

warnings.filterwarnings('ignore')
brand.apply_theme()

PRICE_END = '2026-07-02'
OUT_DIR = Path('../output/4-cross-dataset-investigation')
OUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR = Path('cache/audio')
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path('cache')
print(f'Output directory: {OUT_DIR.resolve()}')

Output directory: G:\congress-trades-investigation\output\4-cross-dataset-investigation


In [2]:
# ── The scoped universe: sectors where federal money is densest ───────────────
TICKER_SECTOR = {
    # defense
    'LMT': 'defense', 'NOC': 'defense', 'RTX': 'defense', 'GD': 'defense',
    'LHX': 'defense', 'PLTR': 'defense', 'BA': 'defense', 'HII': 'defense',
    'LDOS': 'defense', 'KTOS': 'defense', 'AVAV': 'defense', 'TXT': 'defense',
    # pharma / health
    'PFE': 'pharma', 'MRK': 'pharma', 'MRNA': 'pharma', 'JNJ': 'pharma',
    'UNH': 'pharma', 'LLY': 'pharma', 'ABBV': 'pharma', 'CVS': 'pharma',
    # energy
    'XOM': 'energy', 'CVX': 'energy', 'NEE': 'energy', 'CEG': 'energy',
    'OXY': 'energy', 'FSLR': 'energy',
    # federal tech / IT
    'MSFT': 'fedtech', 'AMZN': 'fedtech', 'ORCL': 'fedtech', 'IBM': 'fedtech',
    'ACN': 'fedtech', 'BAH': 'fedtech',
}

# Committees with budget authority / jurisdiction per sector (thomas_id prefixes;
# Appropriations counts everywhere — it moves the actual money).
SECTOR_COMMITTEES = {
    'defense': {'HSAS', 'SSAS', 'HSAP', 'SSAP', 'HLIG', 'SLIN'},
    'pharma':  {'HSIF', 'SSHR', 'SSFI', 'HSAP', 'SSAP', 'HSWM'},
    'energy':  {'HSIF', 'SSEG', 'HSII', 'HSAP', 'SSAP', 'HSSY'},
    'fedtech': {'HSAS', 'SSAS', 'HSGO', 'SSGA', 'HSAP', 'SSAP', 'HLIG', 'SLIN'},
}

# Scoring windows / weights — documented so a viewer can argue with them
LOBBY_WINDOW_DAYS    = 90    # lobbying spend counted within ±90d of the buy
CONTRACT_WINDOW_DAYS = 180   # contract awards counted within 180d after the buy
WEIGHTS = {'timing': 0.30, 'contract_usd': 0.25, 'lobby_usd': 0.20,
           'committee': 0.15, 'trade_usd': 0.10}
print(f'{len(TICKER_SECTOR)} candidate tickers across {len(set(TICKER_SECTOR.values()))} sectors')

32 candidate tickers across 4 sectors


In [3]:
# ── Congress buys (full history, same cleaning as Deliverables 1–2) ───────────
# Names are canonicalized on the raw frame too (not just the cleaned buys): the
# hero timeline filters raw rows by Representative, and Quiver spells one member
# several ways ('Scott Franklin' / 'C. Scott Franklin' / 'Scott Scott Franklin').
raw_congress = data_quiver.canonicalize_names(data_quiver.fetch_congress_trades())
congress_df = data_quiver.clean_congress_buys(raw_congress)
scoped = congress_df[congress_df['Ticker'].isin(TICKER_SECTOR)].copy()
scoped['Sector'] = scoped['Ticker'].map(TICKER_SECTOR)
print(f'{len(scoped):,} buys of the {len(TICKER_SECTOR)} candidate tickers '
      f'by {scoped["Representative"].nunique()} members')

3,543 buys of the 32 candidate tickers by 160 members


In [4]:
# ── Committee membership via BioGuideID (unitedstates/congress-legislators) ───
# Public-domain dataset maintained on GitHub; cached to disk for offline re-runs.
CL_BASE = 'https://unitedstates.github.io/congress-legislators'

def fetch_yaml_cached(name: str) -> object:
    cache_file = CACHE_DIR / f'{name}.pkl'
    if cache_file.exists():
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    resp = requests.get(f'{CL_BASE}/{name}.yaml', timeout=120)
    data = yaml.safe_load(resp.content.decode('utf-8'))  # names contain accents
    with open(cache_file, 'wb') as f:
        pickle.dump(data, f)
    return data

membership = fetch_yaml_cached('committee-membership-current')
committees = fetch_yaml_cached('committees-current')
committee_names = {c['thomas_id']: c['name'] for c in committees}

# bioguide → set of full-committee thomas_ids (sub-committees roll up to parents)
member_committees: dict[str, set] = {}
for thomas_id, members in membership.items():
    parent = thomas_id[:4]
    for m in members:
        if 'bioguide' in m:
            member_committees.setdefault(m['bioguide'], set()).add(parent)

def committee_match(bioguide: str, sector: str) -> list[str]:
    """Full-committee thomas_ids this member sits on with jurisdiction over sector."""
    if not isinstance(bioguide, str):
        return []
    return sorted(member_committees.get(bioguide, set()) & SECTOR_COMMITTEES[sector])

print(f'{len(member_committees)} current members with committee assignments')

528 current members with committee assignments


In [5]:
# ── Lobbying + contract history for each candidate ticker (disk-cached) ───────
def fetch_alt_cached(kind: str, ticker: str) -> pd.DataFrame:
    cache_file = CACHE_DIR / f'{kind}_{ticker}.pkl'
    if cache_file.exists():
        return pd.read_pickle(cache_file)
    fetch = {'lobbying': data_quiver.fetch_lobbying,
             'contracts': data_quiver.fetch_gov_contracts}[kind]
    df = fetch(ticker)
    df.to_pickle(cache_file)
    return df


def normalize_contracts(df: pd.DataFrame) -> pd.DataFrame:
    """Key every award on the date it actually happened, not the date Quiver
    published it. `Date` in govcontractsall is the ingest/publish date — the 2022
    rows are a multi-year backfill (median lag 4+ years), so timing math keyed on
    it would measure Quiver's upload schedule, not contract timing. The real award
    date is USASpending's `action_date`. Exact duplicate rows are dropped.
    """
    df = df.drop_duplicates().copy()
    if 'action_date' in df.columns:
        aw = pd.to_datetime(df['action_date'], errors='coerce')
        df['AwardDate'] = aw.fillna(df['Date'])
    else:
        df['AwardDate'] = df['Date']
    return df.sort_values('AwardDate').reset_index(drop=True)


lobbying, contracts = {}, {}
for t in TICKER_SECTOR:
    lobbying[t] = fetch_alt_cached('lobbying', t)
    contracts[t] = normalize_contracts(fetch_alt_cached('contracts', t))
    span = (f"{contracts[t]['AwardDate'].min():%Y-%m} → {contracts[t]['AwardDate'].max():%Y-%m}"
            if len(contracts[t]) else 'n/a')
    print(f'{t:>5}: {len(lobbying[t]):>6,} lobbying filings | '
          f'{len(contracts[t]):>7,} contract awards ({span})')

  LMT:  3,112 lobbying filings |  64,710 contract awards (2018-01 → 2026-05)


  NOC:  1,391 lobbying filings |   7,447 contract awards (2018-01 → 2026-05)
  RTX:  1,970 lobbying filings |   6,981 contract awards (2018-01 → 2026-07)
   GD:  2,241 lobbying filings |   9,990 contract awards (2018-01 → 2026-06)
  LHX:    377 lobbying filings |   1,323 contract awards (2018-09 → 2026-06)
 PLTR:    531 lobbying filings |     100 contract awards (2018-03 → 2026-06)


   BA:  2,054 lobbying filings |  39,753 contract awards (2018-01 → 2026-06)
  HII:    506 lobbying filings |       9 contract awards (2025-05 → 2025-09)
 LDOS:     35 lobbying filings |       0 contract awards (n/a)
 KTOS:    108 lobbying filings |       2 contract awards (2022-04 → 2022-06)
 AVAV:    264 lobbying filings |      75 contract awards (2018-01 → 2026-06)
  TXT:    530 lobbying filings |   8,721 contract awards (2018-01 → 2026-05)
  PFE:  1,773 lobbying filings |   1,652 contract awards (2018-01 → 2026-06)


  MRK:  1,233 lobbying filings |     103 contract awards (2018-11 → 2026-06)
 MRNA:    124 lobbying filings |      14 contract awards (2022-09 → 2026-06)
  JNJ:  1,348 lobbying filings |  16,904 contract awards (2018-01 → 2026-06)
  UNH:    892 lobbying filings |       0 contract awards (n/a)
  LLY:     86 lobbying filings |       3 contract awards (2025-09 → 2026-05)
 ABBV:    465 lobbying filings |      45 contract awards (2018-01 → 2026-01)
  CVS:    505 lobbying filings |       4 contract awards (2022-09 → 2025-07)


  XOM:    875 lobbying filings |     406 contract awards (2018-01 → 2022-07)
  CVX:    752 lobbying filings |     580 contract awards (2018-01 → 2026-01)
  NEE:  1,090 lobbying filings |      34 contract awards (2025-07 → 2026-06)
  CEG:    219 lobbying filings |       5 contract awards (2018-02 → 2025-06)
  OXY:    334 lobbying filings |       0 contract awards (n/a)
 FSLR:    194 lobbying filings |       0 contract awards (n/a)
 MSFT:  2,246 lobbying filings |     288 contract awards (2018-01 → 2026-06)
 AMZN:    646 lobbying filings |       0 contract awards (n/a)


 ORCL:  1,643 lobbying filings |      69 contract awards (2018-02 → 2026-06)
  IBM:     50 lobbying filings |     222 contract awards (2023-01 → 2026-07)
  ACN:    262 lobbying filings |     296 contract awards (2023-01 → 2026-06)
  BAH:    108 lobbying filings |     651 contract awards (2023-01 → 2026-06)


In [6]:
# ── Score every scoped buy on alignment, dollars, and committee relevance ─────
price_cache = data_massive.load_price_cache(
    ['SPY'] + sorted(TICKER_SECTOR), '2012-06-01', PRICE_END, verbose=False)

rows = []
for _, buy in scoped.iterrows():
    t, d, sector = buy['Ticker'], buy['TransactionDate'], buy['Sector']
    lob, con = lobbying[t], contracts[t]

    lob_win = lob[(lob['Date'] >= d - timedelta(days=LOBBY_WINDOW_DAYS)) &
                  (lob['Date'] <= d + timedelta(days=LOBBY_WINDOW_DAYS))]
    con_win = con[(con['AwardDate'] > d) &
                  (con['AwardDate'] <= d + timedelta(days=CONTRACT_WINDOW_DAYS))]
    nxt = con_win.nlargest(1, 'Amount')

    px = price_cache.get(t, pd.Series(dtype=float))
    p0 = data_massive.get_price_on_or_after(px, d, max_gap_days=7)
    p1 = data_massive.get_price_on_or_after(px, d + timedelta(days=120))
    move = p1 / p0 - 1.0 if p0 == p0 and p1 == p1 else np.nan

    matches = committee_match(buy.get('BioGuideID'), sector)
    rows.append({
        'Representative': buy['Representative'], 'BioGuideID': buy.get('BioGuideID'),
        'Ticker': t, 'Sector': sector, 'BuyDate': d, 'ReportDate': buy['ReportDate'],
        'TradeUSD': buy['Amount'],
        'Committees': '; '.join(committee_names.get(c, c) for c in matches),
        'CommitteeMatch': bool(matches),
        'LobbyUSD_±90d': float(lob_win['Amount'].sum()),
        'LobbyFilings_±90d': int(len(lob_win)),
        'ContractUSD_180d': float(con_win['Amount'].sum()),
        'ContractCount_180d': int(len(con_win)),
        'DaysToLargestAward': int((nxt['AwardDate'].iloc[0] - d).days) if not nxt.empty else np.nan,
        'LargestAwardUSD': float(nxt['Amount'].iloc[0]) if not nxt.empty else np.nan,
        'LargestAwardAgency': str(nxt['Agency'].iloc[0]) if not nxt.empty else '',
        'StockMove_120d': move,
    })

buys_scored = pd.DataFrame(rows)

def minmax(s):
    s = s.fillna(0.0)
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s * 0.0

buys_scored['Score'] = (
    WEIGHTS['timing'] * np.exp(-buys_scored['DaysToLargestAward'].fillna(9e9) / 90.0)
    + WEIGHTS['contract_usd'] * minmax(np.log10(1 + buys_scored['ContractUSD_180d']))
    + WEIGHTS['lobby_usd'] * minmax(np.log10(1 + buys_scored['LobbyUSD_±90d']))
    + WEIGHTS['committee'] * buys_scored['CommitteeMatch'].astype(float)
    + WEIGHTS['trade_usd'] * minmax(np.log10(1 + buys_scored['TradeUSD'].fillna(0)))
)
print(f'{len(buys_scored):,} scored buys | committee-matched: {buys_scored["CommitteeMatch"].mean():.0%}')

3,543 scored buys | committee-matched: 31%


In [7]:
# ── Aggregate to (representative, ticker) cases and rank ─────────────────────
cases = (
    buys_scored
    .groupby(['Representative', 'Ticker'], as_index=False)
    .agg(
        Sector=('Sector', 'first'),
        Committees=('Committees', 'first'),
        CommitteeMatch=('CommitteeMatch', 'first'),
        NumBuys=('BuyDate', 'count'),
        FirstBuy=('BuyDate', 'min'),
        LastBuy=('BuyDate', 'max'),
        TotalTradeUSD=('TradeUSD', 'sum'),
        BestScore=('Score', 'max'),
        AvgScore=('Score', 'mean'),
        BestBuyDate=('Score', 'idxmax'),
    )
)
cases['BestBuyDate'] = buys_scored.loc[cases['BestBuyDate'], 'BuyDate'].values
detail = buys_scored.set_index(['Representative', 'Ticker', 'BuyDate'])
best_rows = buys_scored.sort_values('Score', ascending=False).drop_duplicates(
    subset=['Representative', 'Ticker'])
cases = cases.merge(
    best_rows[['Representative', 'Ticker', 'LobbyUSD_±90d', 'ContractUSD_180d',
               'DaysToLargestAward', 'LargestAwardUSD', 'LargestAwardAgency',
               'StockMove_120d']],
    on=['Representative', 'Ticker'])
cases = cases.sort_values('BestScore', ascending=False).reset_index(drop=True)

# Confidence: needs a committee match AND a real contract in window to be worth
# hand-verification; otherwise it's just a popular stock.
def confidence(r):
    if r['CommitteeMatch'] and r['LargestAwardUSD'] > 1e7 and r['DaysToLargestAward'] <= 120:
        return 'HIGH — verify against public record'
    if r['CommitteeMatch'] and r['ContractUSD_180d'] > 0:
        return 'MEDIUM — alignment present, award size/timing weaker'
    return 'LOW — no committee link; likely just a widely-held stock'

cases['Confidence'] = cases.apply(confidence, axis=1)
cases.to_csv(OUT_DIR / 'candidates.csv', index=False)
buys_scored.sort_values('Score', ascending=False).to_csv(
    OUT_DIR / 'scored_buys_full.csv', index=False)

pd.set_option('display.width', 200)
cols = ['Representative', 'Ticker', 'Sector', 'BestBuyDate', 'Committees',
        'LobbyUSD_±90d', 'LargestAwardUSD', 'DaysToLargestAward',
        'StockMove_120d', 'BestScore', 'Confidence']
cases[cols].head(10)

,Representative,Ticker,Sector,BestBuyDate,Committees,LobbyUSD_±90d,LargestAwardUSD,DaysToLargestAward,StockMove_120d,BestScore,Confidence
0,Ro Khanna,LMT,defense,2019-01-18,House Committee on Armed Services,17423563.0,5.280450e+08,6.0,0.209071,0.885492,HIGH — verify against public record
1,Gilbert Cisneros,PLTR,defense,2026-04-14,House Committee on Armed Services,5455000.0,9.468781e+07,3.0,NaN,0.880434,HIGH — verify against public record
2,William R. Keating,NOC,defense,2021-03-17,House Committee on Armed Services,7855000.0,2.919305e+08,7.0,0.172425,0.868610,HIGH — verify against public record
3,Ro Khanna,NOC,defense,2019-06-03,House Committee on Armed Services,9255000.0,1.947473e+08,4.0,0.198247,0.861709,HIGH — verify against public record
4,C. Scott Franklin,LMT,defense,2022-09-12,House Committee on Appropriations,11242940.0,5.237555e+07,1.0,0.104266,0.859294,HIGH — verify against public record
5,Ro Khanna,MSFT,fedtech,2023-09-06,House Committee on Armed Services; House Commi...,8855000.0,1.603678e+07,5.0,0.105323,0.856058,HIGH — verify against public record
6,Lois Frankel,GD,defense,2022-07-20,House Committee on Appropriations,4137500.0,5.625200e+08,8.0,0.137505,0.852823,HIGH — verify against public record
7,Josh Gottheimer,MSFT,fedtech,2021-10-28,House Permanent Select Committee on Intelligence,7380000.0,2.374492e+06,1.0,-0.083367,0.846242,"MEDIUM — alignment present, award size/timing ..."
8,Ro Khanna,BA,defense,2019-06-03,House Committee on Armed Services,9280000.0,2.952822e+08,10.0,0.106377,0.844630,HIGH — verify against public record
9,Ro Khanna,GD,defense,2018-11-13,House Committee on Armed Services,7724692.0,7.728887e+07,14.0,-0.046424,0.838661,HIGH — verify against public record


In [8]:
# ── The hero visual: three stacked, synchronized timelines on one ticker ──────
# Top: Congress buys/sells. Middle: quarterly lobbying spend. Bottom: contract
# awards (on AwardDate — when the award happened, not when Quiver published it).
# Panels appear in sequence, then the alignment window is highlighted.
def animate_three_timelines(ticker, case, output_path,
                            phase_s=(6.0, 4.0, 4.0), reveal_s=3.0, hold_s=3.0,
                            audio_wav=None, show_alignment=True, title=None):
    fps = brand.VIDEO_FPS
    rep = case['Representative']
    buy_date = pd.Timestamp(case['BestBuyDate'])
    w0 = buy_date - pd.DateOffset(months=15)
    w1 = buy_date + pd.DateOffset(months=18)

    # -- data in window ---------------------------------------------------------
    all_tx = raw_congress.dropna(subset=['TransactionDate', 'Ticker'])
    tx = all_tx[(all_tx['Ticker'] == ticker) &
                (all_tx['TransactionDate'] >= w0) & (all_tx['TransactionDate'] <= w1)]
    tx_buys = tx[tx['Transaction'].str.contains('Purchase', case=False, na=False)]
    tx_sells = tx[tx['Transaction'].str.contains('Sale', case=False, na=False)]
    rep_buys = tx_buys[tx_buys['Representative'] == rep]

    lob = lobbying[ticker]
    lob_w = lob[(lob['Date'] >= w0) & (lob['Date'] <= w1)]
    lob_q = lob_w.groupby(pd.Grouper(key='Date', freq='QS'))['Amount'].sum()

    con = contracts[ticker]
    con_w = con[(con['AwardDate'] >= w0) & (con['AwardDate'] <= w1)]
    con_d = con_w.groupby('AwardDate')['Amount'].sum()  # stack same-day awards
    # The award the SCORE found: largest within the 180-day scoring window,
    # so the alignment band matches candidates.csv (not the widest award on screen).
    con_180 = con[(con['AwardDate'] > buy_date) &
                  (con['AwardDate'] <= buy_date + timedelta(days=CONTRACT_WINDOW_DAYS))]
    biggest = (con_180 if not con_180.empty else con_w).nlargest(1, 'Amount').iloc[0]

    # -- figure -----------------------------------------------------------------
    fig, axes = plt.subplots(3, 1, figsize=(brand.FIG_W, brand.FIG_H),
                             facecolor=BG, sharex=True,
                             gridspec_kw={'hspace': 0.32})
    fig.suptitle(title if title else ticker,
                 color=GREEN, fontsize=21, fontfamily=VCR, fontweight='bold', y=0.965)
    panel_titles = ['CONGRESS TRADES', 'LOBBYING SPEND (QUARTERLY)',
                    'FEDERAL CONTRACT AWARDS']
    for ax, pt in zip(axes, panel_titles):
        ax.set_facecolor(BG)
        ax.set_title(pt, color=OFF_WHITE, fontsize=12, fontfamily=VCR,
                     loc='left', pad=8)
        ax.grid(True, color=GRID, linewidth=0.5, alpha=0.5, zorder=0)
        ax.set_axisbelow(True)
        for s in ax.spines.values():
            s.set_edgecolor(GRID)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_xlim(w0, w1)
        ax.tick_params(labelsize=10)

    def fmt_usd(x, _):
        if x >= 1e9: return f'${x/1e9:.0f}B'
        if x >= 1e6: return f'${x/1e6:.0f}M'
        if x >= 1e3: return f'${x/1e3:.0f}K'
        return f'${x:.0f}'

    # Panel 1: trades as amount markers (log y — trade sizes span decades)
    ax1 = axes[0]
    amt_all = pd.concat([tx_buys['Amount'], tx_sells['Amount']]).dropna()
    ax1.set_yscale('log')
    ax1.set_ylim(max(amt_all.min() * 0.4, 500), amt_all.max() * 8)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(fmt_usd))
    buys_sc = ax1.scatter(tx_buys['TransactionDate'], tx_buys['Amount'],
                          marker='^', s=90, color=GREEN, alpha=0.0, zorder=3)
    sells_sc = ax1.scatter(tx_sells['TransactionDate'], tx_sells['Amount'],
                           marker='v', s=90, color=RED, alpha=0.0, zorder=3)
    rep_sc = ax1.scatter(rep_buys['TransactionDate'], rep_buys['Amount'],
                         marker='^', s=260, facecolors='none', edgecolors=GREEN,
                         linewidths=2.0, alpha=0.0, zorder=4)
    rep_last = str(rep).split()[-1].upper()
    y_note = float(rep_buys['Amount'].max()) if rep_buys['Amount'].notna().any() else 15000.0
    rep_note = ax1.annotate(f'{rep_last} BUYS', xy=(buy_date, y_note),
                            xytext=(12, 22), textcoords='offset points', color=GREEN,
                            fontsize=13, fontfamily=VCR, alpha=0.0, zorder=5)

    # Panel 2: quarterly lobbying bars
    ax2 = axes[1]
    ax2.set_ylim(0, max(lob_q.max() * 1.25, 1))
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(fmt_usd))
    lob_bars = ax2.bar(lob_q.index, [0] * len(lob_q), width=80, align='edge',
                       color=OLIVE, edgecolor=GREEN, linewidth=0.8, zorder=3)

    # Panel 3: contract awards as stems (dollar labels, not 10^x — set after log scale)
    ax3 = axes[2]
    ax3.set_yscale('log')
    ax3.set_ylim(max(con_d.min() * 0.4, 1e4), con_d.max() * 12)
    ax3.yaxis.set_major_formatter(plt.FuncFormatter(fmt_usd))
    ax3.yaxis.set_minor_formatter(plt.NullFormatter())
    con_lines = ax3.vlines(con_d.index, ymin=ax3.get_ylim()[0], ymax=con_d.values,
                           color=GREEN, linewidth=1.3, alpha=0.0, zorder=3)
    big_note = ax3.annotate(
        f"{fmt_usd(biggest['Amount'], None)} - {str(biggest['Agency'])[:34].upper()}",
        xy=(biggest['AwardDate'], biggest['Amount']), xytext=(10, 10),
        textcoords='offset points', color=GREEN, fontsize=12, fontfamily=VCR,
        alpha=0.0, zorder=5)
    big_i = con_d.index.get_loc(pd.Timestamp(biggest['AwardDate']))
    awards_start = con['AwardDate'].min()
    if awards_start > w0:
        ax3.text(0.01, 0.88, f'AWARDS DATA BEGINS {awards_start:%b %Y}',
                 transform=ax3.transAxes, color=OFF_WHITE, alpha=0.45,
                 fontsize=10, fontfamily=VCR)

    # Alignment band: buy date → the scored award, across all three panels
    award_date = pd.Timestamp(biggest['AwardDate'])
    spans = [ax.axvspan(buy_date, award_date, color=GREEN, alpha=0.0, zorder=1)
             for ax in axes]
    days_lbl = axes[0].annotate(
        f"{(award_date - buy_date).days} DAY{'S' if (award_date - buy_date).days != 1 else ''}",
        xy=(award_date, ax1.get_ylim()[1] * 0.35),
        xytext=(-8, 0), textcoords='offset points', color=GREEN, fontsize=14,
        fontfamily=VCR, ha='right', alpha=0.0, zorder=6)

    # -- animation schedule ------------------------------------------------------
    p1, p2, p3 = (int(s * fps) for s in phase_s)
    f_p2, f_p3 = p1, p1 + p2
    f_reveal = p1 + p2 + p3
    n_total = f_reveal + int((reveal_s + hold_s) * fps)

    t_all = mdates.date2num(pd.to_datetime([w0, w1]).to_pydatetime())
    tx_b_x = mdates.date2num(pd.to_datetime(tx_buys['TransactionDate']).dt.to_pydatetime())
    tx_s_x = mdates.date2num(pd.to_datetime(tx_sells['TransactionDate']).dt.to_pydatetime())
    rep_x = mdates.date2num(pd.to_datetime(rep_buys['TransactionDate']).dt.to_pydatetime())
    con_x = mdates.date2num(pd.to_datetime(con_d.index).to_pydatetime())

    def sweep_alpha(xs, frame, f0, nf):
        """Per-point alpha: events appear as a sweep line passes them."""
        if frame < f0:
            return np.zeros(len(xs))
        prog = min((frame - f0) / max(nf - 1, 1), 1.0)
        x_now = t_all[0] + prog * (t_all[1] - t_all[0])
        return np.clip((x_now - xs) / ((t_all[1] - t_all[0]) * 0.02), 0, 1)

    def update(frame):
        # A collection's scalar alpha overrides per-color alphas — clear it before
        # animating via RGBA lists, or everything stays invisible.
        for coll in (buys_sc, sells_sc, rep_sc, con_lines):
            coll.set_alpha(None)
        a_b = sweep_alpha(tx_b_x, frame, 0, p1)
        buys_sc.set_facecolor([(0, 246/255, 0, a) for a in a_b])
        a_s = sweep_alpha(tx_s_x, frame, 0, p1)
        sells_sc.set_facecolor([(1, 0, 0, a) for a in a_s])
        a_r = sweep_alpha(rep_x, frame, 0, p1)
        rep_sc.set_edgecolor([(0, 246/255, 0, a) for a in a_r])
        rep_note.set_alpha(float(a_r.max()) if len(a_r) else 0.0)

        t2 = animation.smooth_step((frame - f_p2) / max(p2 * 0.6, 1))
        for bar, v in zip(lob_bars, lob_q.values):
            bar.set_height(v * t2)

        a_c = sweep_alpha(con_x, frame, f_p3, p3)
        con_lines.set_color([(0, 246/255, 0, a) for a in a_c])
        big_note.set_alpha((float(a_c[big_i]) if len(a_c) else 0.0) if show_alignment else 0.0)

        t4 = animation.smooth_step((frame - f_reveal) / (0.8 * fps))
        for sp in spans:
            sp.set_alpha((0.10 * t4) if show_alignment else 0.0)
        days_lbl.set_alpha(t4 if show_alignment else 0.0)

        return ([buys_sc, sells_sc, rep_sc, rep_note, con_lines, big_note, days_lbl]
                + list(lob_bars) + spans)

    ani = mpl_animation.FuncAnimation(fig, update, frames=n_total,
                                      interval=1000 / fps, blit=True)
    print(f'Rendering {output_path.name}  ({n_total} frames @ {fps} fps)...')
    animation.save_animation(ani, fig, output_path, audio_wav)
    return n_total / fps

print('Hero timeline builder ready')

Hero timeline builder ready


In [9]:
# ── Render the hero + a second ticker (different sector, same pattern) ────────
# HERO/SECOND default to the ranking, restricted to hand-verifiable candidates.
# Override the ticker strings here after manual verification if needed.
verifiable = cases[cases['Confidence'].str.startswith(('HIGH', 'MEDIUM'))]
# HERO override: the auto-ranked #1 (Ro Khanna / LMT) FAILED a primary-source fact-check
# (trade was his spouse's independently-managed trust; the "$528M contract" was a wrong-year
# conflation). Manually verified replacement = Scott Franklin / LMT: self-owned (House Clerk
# PTR #20021836, own Traditional+Roth IRA), on House Armed Services in 2022 (H. Rept. 117-666).
# The committee string is corrected off the current-congress default (he moved to Appropriations
# only in 2023). Rendered with show_alignment=False so the visual makes NO "contract N days
# later" causal claim (LMT books thousands of awards/yr -> proximity is noise).
hero_case = cases[(cases['Representative'].str.contains('Franklin', case=False)) &
                  (cases['Ticker'] == 'LMT')].iloc[0].copy()
hero_case['Committees'] = 'House Committee on Armed Services (117th Cong., verified H. Rept. 117-666)'
second_pool = verifiable[verifiable['Sector'] != hero_case['Sector']]
if second_pool.empty:
    second_pool = verifiable.iloc[1:]
high = second_pool[second_pool['Confidence'].str.startswith('HIGH')]
second_case = (high if not high.empty else second_pool).iloc[0]

print(f"HERO:   {hero_case['Ticker']} — {hero_case['Representative']} "
      f"({hero_case['Confidence']})")
print(f"SECOND: {second_case['Ticker']} — {second_case['Representative']} "
      f"({second_case['Confidence']})")

PHASES, REVEAL_S, HOLD_S = (6.0, 4.0, 4.0), 3.0, 3.0
total_s = sum(PHASES) + REVEAL_S + HOLD_S
hero_wav = render_track(
    [Cue('keystroke', 0.1, gain_db=-4)]
    + ticks_every('tick', 0.5, PHASES[0] - 0.5, 0.9, gain_db=-10)
    + [Cue('keystroke', PHASES[0], gain_db=-4),
       Cue('bar_grow', PHASES[0] + 0.2, gain_db=-6),
       Cue('keystroke', PHASES[0] + PHASES[1], gain_db=-4)]
    + ticks_every('tick', PHASES[0] + PHASES[1] + 0.4, sum(PHASES) - 0.4, 0.7, gain_db=-10)
    + [Cue('line_swell', sum(PHASES) - 0.5, gain_db=-3),
       Cue('resolve_tone', sum(PHASES) + REVEAL_S, gain_db=-5)],
    duration_s=total_s,
    out_path=AUDIO_DIR / 'd4_hero.wav',
)
animate_three_timelines(hero_case['Ticker'], hero_case,
                        OUT_DIR / f"hero_timeline_{hero_case['Ticker']}.mp4",
                        phase_s=PHASES, reveal_s=REVEAL_S, hold_s=HOLD_S,
                        audio_wav=hero_wav, show_alignment=False, title='Lockheed Martin Corp')

HERO:   LMT — Ro Khanna (HIGH — verify against public record)
SECOND: MSFT — Ro Khanna (HIGH — verify against public record)


findfont: Failed to find font weight bold, now using 400.


Rendering hero_timeline_LMT.mp4  (1200 frames @ 60 fps)...


Saved hero_timeline_LMT.mp4  (0.8 MB)


20.0

In [10]:
# Second ticker — shorter walkthrough ("this isn't a one-off")
PHASES2, REVEAL2_S, HOLD2_S = (4.0, 2.5, 2.5), 2.0, 2.5
total2_s = sum(PHASES2) + REVEAL2_S + HOLD2_S
second_wav = render_track(
    [Cue('keystroke', 0.1, gain_db=-4),
     Cue('bar_grow', PHASES2[0] + 0.2, gain_db=-6),
     Cue('keystroke', PHASES2[0] + PHASES2[1], gain_db=-4),
     Cue('line_swell', sum(PHASES2) - 0.4, gain_db=-4),
     Cue('resolve_tone', sum(PHASES2) + REVEAL2_S, gain_db=-6)],
    duration_s=total2_s,
    out_path=AUDIO_DIR / 'd4_second.wav',
)
RENDER_SECOND = False  # reframed script uses the "run the scan across everyone" line instead
# of a 2nd specific example. Only set True AFTER primary-source-verifying a second member/ticker.
if RENDER_SECOND:
    animate_three_timelines(second_case['Ticker'], second_case,
                            OUT_DIR / f"second_timeline_{second_case['Ticker']}.mp4",
                            phase_s=PHASES2, reveal_s=REVEAL2_S, hold_s=HOLD2_S,
                            audio_wav=second_wav, show_alignment=False)
else:
    print('Second timeline skipped (no verified 2nd example; script uses the systemic-scan line).')

Rendering second_timeline_MSFT.mp4  (810 frames @ 60 fps)...


Saved second_timeline_MSFT.mp4  (0.8 MB)


13.5

In [11]:
# ── Headline numbers + the hand-verification handoff ─────────────────────────
def case_dict(c):
    return {
        'ticker': str(c['Ticker']), 'representative': str(c['Representative']),
        'sector': str(c['Sector']), 'committees': str(c['Committees']),
        'buy_date': str(pd.Timestamp(c['BestBuyDate']).date()),
        'lobby_usd_pm90d': float(c['LobbyUSD_±90d']),
        'largest_award_usd': float(c['LargestAwardUSD']),
        'largest_award_agency': str(c['LargestAwardAgency']),
        'days_buy_to_award': (None if pd.isna(c['DaysToLargestAward'])
                              else int(c['DaysToLargestAward'])),
        'stock_move_120d': (None if pd.isna(c['StockMove_120d'])
                            else float(c['StockMove_120d'])),
        'confidence': str(c['Confidence']),
    }

summary = {
    'method': {
        'universe_tickers': len(TICKER_SECTOR),
        'scoped_buys': int(len(buys_scored)),
        'lobby_window_days': LOBBY_WINDOW_DAYS,
        'contract_window_days': CONTRACT_WINDOW_DAYS,
        'weights': WEIGHTS,
        'award_date_note': 'All award timing is keyed on USASpending action_date (the '
                           'date the contract action happened). Quiver\'s Date column '
                           'is its publish/ingest date — its 2022 rows are a multi-year '
                           'backfill, so timing keyed on Date would be an ingest artifact.',
        'committee_source': 'unitedstates/congress-legislators (public domain)',
        'committee_note': 'Membership is CURRENT-congress only, so old buys are scored '
                          'against today\'s seats — fine for recent cases, wrong for '
                          'former members. Verify the seat held on the buy date.',
    },
    'hero_case': case_dict(hero_case),
    'second_case': case_dict(second_case),
    'shortlist': [case_dict(c) for _, c in cases.head(8).iterrows()],
    'caveat': 'Every case is legal, public activity. Verify each shortlist row against '
              'the Senate LDA database, USASpending.gov, and the original disclosure '
              'before committing any name/ticker to the script.',
}
with open(OUT_DIR / 'summary_stats.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Shortlist for manual verification:')
for i, c in cases.head(8).iterrows():
    print(f"{i+1}. {c['Ticker']:>5} — {c['Representative']:<28} "
          f"buy {pd.Timestamp(c['BestBuyDate']).date()} | "
          f"award {c['LargestAwardUSD']/1e6:,.0f}M in {c['DaysToLargestAward']:.0f}d | "
          f"{c['Confidence']}")

Shortlist for manual verification:
1.   LMT — Ro Khanna                    buy 2019-01-18 | award 528M in 6d | HIGH — verify against public record
2.  PLTR — Gilbert Cisneros             buy 2026-04-14 | award 95M in 3d | HIGH — verify against public record
3.   NOC — William R. Keating           buy 2021-03-17 | award 292M in 7d | HIGH — verify against public record
4.   NOC — Ro Khanna                    buy 2019-06-03 | award 195M in 4d | HIGH — verify against public record
5.   LMT — C. Scott Franklin            buy 2022-09-12 | award 52M in 1d | HIGH — verify against public record
6.  MSFT — Ro Khanna                    buy 2023-09-06 | award 16M in 5d | HIGH — verify against public record
7.    GD — Lois Frankel                 buy 2022-07-20 | award 563M in 8d | HIGH — verify against public record
8.  MSFT — Josh Gottheimer              buy 2021-10-28 | award 2M in 1d | MEDIUM — alignment present, award size/timing weaker
